
# 🔍 NB04 — TF-IDF + BM25 Search Index
## Investor Intelligence Platform — FIIs Brasil 🇧🇷
### CRISP-DM Phase: Modeling

| | |
|---|---|
| **Input** | `data/silver/silver_articles.parquet` |
| **Output** | `data/gold/tfidf_bm25/` (4 artefatos + funções de busca) |
| **Engines** | `sklearn.TfidfVectorizer` + `rank_bm25.BM25Okapi` |

## 🎯 O que este notebook faz

1. **Corpus** — tokeniza `text_clean` do Silver em listas de tokens normalizados.
2. **TF-IDF** — treina `TfidfVectorizer`, persiste como scipy sparse matrix (`.npz`).
3. **BM25** — treina `BM25Okapi(k1=1.5, b=0.75)`, persiste com `pickle`.
4. **Funções de busca** — `search_tfidf(query, top_k)` e `search_bm25(query, top_k)` utilizáveis em NB05/NB06/NB07.
5. **Combina** — `search_hybrid(query, top_k, alpha=0.4)` = 0.4×TF-IDF + 0.6×BM25 (MinMax normalizado).

## 📐 Fórmulas

### TF-IDF
```
TF-IDF(t,d) = TF(t,d) × log((N+1) / (df(t)+1)) + 1
```

### BM25 (Robertson & Zaragoza, 2009)
```
BM25(D,Q) = Σ IDF(qi) · [f(qi,D)·(k1+1)] / [f(qi,D) + k1·(1 − b + b·|D|/avgdl)]
k1=1.5  b=0.75
```

### Hybrid Score
```
score_hybrid = α · score_tfidf_norm + (1−α) · score_bm25_norm      α=0.4
```

## 📦 Seção 1 — Imports

In [1]:
import sys, os, pickle, warnings
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import re, random, unicodedata
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
import pyarrow as pa

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import nltk

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

warnings.filterwarnings("ignore")
print(f"Python     : {sys.version.split()[0]}")
print(f"PySpark    : {pyspark.__version__}")
print(f"sklearn    : {__import__('sklearn').__version__}")
print(f"rank_bm25  : rank_bm25 library loaded (no __version__ attribute)")

Python     : 3.12.12
PySpark    : 4.1.2
sklearn    : 1.9.0
rank_bm25  : rank_bm25 library loaded (no __version__ attribute)


## ⚙️ Seção 2 — Configuração e SparkSession

In [2]:
PROJECT_ROOT = Path(os.getcwd()).resolve()
_lim = 6
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT and _lim > 0:
    PROJECT_ROOT = PROJECT_ROOT.parent; _lim -= 1
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import RANDOM_SEED
from src.utils.logger import ingestion_logger, log_quality_check, log_spark_start

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SILVER_DIR     = PROJECT_ROOT / "data" / "silver"
GOLD_INDEX_DIR = PROJECT_ROOT / "data" / "gold" / "tfidf_bm25"
GOLD_INDEX_DIR.mkdir(parents=True, exist_ok=True)

SPARK_DRIVER_MEMORY = "4g"
SPARK_APP_NAME = "fii_platform"
SPARK_SHUFFLE_PARTS = 200

spark = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}_index")
    .master("local[*]")
    .config("spark.driver.memory",          SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTS)
    .config("spark.sql.adaptive.enabled",   "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

logger = ingestion_logger()
log_spark_start(logger, spark.sparkContext.appName, spark.version)

for _res in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"corpora/{_res}")
    except LookupError:
        try: nltk.download(_res, quiet=True)
        except: pass
try:
    from nltk.corpus import stopwords as _nltk_sw
    STOPWORDS_PT = set(_nltk_sw.words("portuguese"))
except Exception:
    STOPWORDS_PT = set()

print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"GOLD_INDEX_DIR : {GOLD_INDEX_DIR}")
print(f"Spark          : {spark.sparkContext.appName} | {spark.version}")
print(f"Stopwords PT   : {len(STOPWORDS_PT)} termos")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 15:48:12 WARN Utils: Your hostname, MacBook-Air-100.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.4 instead (on interface en0)
26/06/14 15:48:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:48:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 15:48:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/14 15:48:13 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


2026-06-14T15:48:15 | INFO     | fii_pipeline.ingestion | SPARK_START | app=fii_platform_index | spark=4.1.2
PROJECT_ROOT   : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform
GOLD_INDEX_DIR : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/gold/tfidf_bm25
Spark          : fii_platform_index | 4.1.2
Stopwords PT   : 207 termos


## 📂 Seção 3 — Carregar Silver e Construir Corpus

In [3]:

SILVER_PATH = SILVER_DIR / "silver_articles.parquet"
if not SILVER_PATH.exists():
    raise FileNotFoundError(f"Silver não encontrado: {SILVER_PATH}\nExecute NB02 primeiro.")

df = pd.read_parquet(SILVER_PATH)
df = df[df.get("has_content", pd.Series([True]*len(df))) != False].copy()
df["text_clean"] = df["text_clean"].fillna("").astype(str)
df = df[df["text_clean"].str.strip().str.len() > 0].reset_index(drop=True)

print(f"Silver carregado : {len(df):,} documentos")
print(f"Colunas          : {list(df.columns)}")
print(f"\nsource_type:\n{df['source_type'].value_counts().to_string()}")

Silver carregado : 126 documentos
Colunas          : ['article_id', 'source', 'source_type', 'source_label', 'title', 'url', 'author', 'published_at', 'published_dt', 'collected_at', 'language', 'tags', 'query_used', 'ingestion_method', 'text_clean', 'word_count', 'char_count', 'has_content', 'content_hash', 'metadata_json']

source_type:
source_type
rss         69
scraping    55
reddit       2


## 🔤 Seção 4 — Tokenizador PT-BR

In [4]:

_TOKEN_RE = re.compile(r"[a-zà-ü]{3,}", re.IGNORECASE | re.UNICODE)

def _norm(s: str) -> str:
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode("ascii").lower()

def tokenize_pt(text: str) -> List[str]:
    """Tokenizador PT-BR: lowercase → regex → NFD normalize → remove stopwords."""
    if not text or not isinstance(text, str):
        return []
    tokens = [_norm(t) for t in _TOKEN_RE.findall(text.lower())]
    return [t for t in tokens if t not in STOPWORDS_PT and len(t) >= 3]

def tokens_to_str(tokens: List[str]) -> str:
    return " ".join(tokens)

# Construir corpus tokenizado
corpus_tokens: List[List[str]] = [tokenize_pt(t) for t in df["text_clean"].tolist()]
corpus_strings: List[str]      = [tokens_to_str(t) for t in corpus_tokens]

n_empty = sum(1 for t in corpus_tokens if len(t) == 0)
all_vocab = {tok for tokens in corpus_tokens for tok in tokens}

print(f"Corpus: {len(corpus_tokens):,} documentos tokenizados")
print(f"  Documentos vazios: {n_empty}")
print(f"  Vocabulário único: {len(all_vocab):,} termos")
print(f"  Tokens/doc médio : {np.mean([len(t) for t in corpus_tokens]):.1f}")

# Fallback: docs vazios recebem tokens do título
for i, toks in enumerate(corpus_tokens):
    if not toks:
        corpus_tokens[i] = tokenize_pt(str(df.at[i, "title"]))
        corpus_strings[i] = tokens_to_str(corpus_tokens[i])

# Salvar mapeamento doc_index → article_id
doc_map = pd.DataFrame({
    "doc_index":  range(len(df)),
    "article_id": df["article_id"].tolist(),
    "source":     df["source"].tolist(),
    "title":      df["title"].tolist(),
})
doc_map.to_parquet(GOLD_INDEX_DIR / "doc_index_map.parquet", index=False, compression="snappy")
print(f"\n✅ doc_index_map.parquet salvo ({len(doc_map):,} entradas)")

Corpus: 126 documentos tokenizados
  Documentos vazios: 0
  Vocabulário único: 7,752 termos
  Tokens/doc médio : 360.8

✅ doc_index_map.parquet salvo (126 entradas)



## 🧮 Seção 5 — TF-IDF Vectorizer

`TfidfVectorizer` com:
- `ngram_range=(1,2)` — unigramas + bigramas
- `max_features=50_000` — limita vocabulário
- `sublinear_tf=True` — aplica log(1+tf) para suavizar termos frequentes
- `min_df=2` — descarta termos que aparecem em só 1 doc

In [5]:

print("🧮 Treinando TF-IDF Vectorizer...")

tfidf_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    token_pattern=r"[a-zà-ü]{3,}",
    strip_accents=None,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_strings)

feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"  TF-IDF matrix shape : {tfidf_matrix.shape}")
print(f"  Vocabulário:          {len(feature_names):,} termos/bigramas")
print(f"  Sparsity:             {100*(1 - tfidf_matrix.nnz / np.prod(tfidf_matrix.shape)):.2f}%")
print(f"  Nnz:                  {tfidf_matrix.nnz:,}")

# Salvar
sp.save_npz(str(GOLD_INDEX_DIR / "tfidf_matrix.npz"), tfidf_matrix)
with open(GOLD_INDEX_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

print(f"\n✅ tfidf_matrix.npz + tfidf_vectorizer.pkl salvos")

🧮 Treinando TF-IDF Vectorizer...
  TF-IDF matrix shape : (126, 9449)
  Vocabulário:          9,449 termos/bigramas
  Sparsity:             96.62%
  Nnz:                  40,266

✅ tfidf_matrix.npz + tfidf_vectorizer.pkl salvos



## 🔎 Seção 6 — BM25 Index (rank_bm25)

`BM25Okapi` com parâmetros recomendados por Robertson & Zaragoza (2009):
- `k1 = 1.5` — saturação de frequência de termos
- `b = 0.75` — normalização por comprimento do documento

O índice opera sobre a lista de listas de tokens (sem join),
respeitando o esquema nativo do BM25.

In [6]:

print("🔎 Construindo BM25 Index (rank_bm25)...")

bm25_index = BM25Okapi(
    corpus_tokens,
    k1=1.5,
    b=0.75,
)


print(f"  BM25 corpus size : {bm25_index.corpus_size:,}")
print(f"  BM25 avgdl       : {bm25_index.avgdl:.2f} tokens")
print(f"  IDF vocab size   : {len(bm25_index.idf):,}")

# Salvar
with open(GOLD_INDEX_DIR / "bm25_index.pkl", "wb") as f:
    pickle.dump(bm25_index, f)

# Salvar corpus tokenizado (necessário para reconstrução futura)
with open(GOLD_INDEX_DIR / "corpus_tokens.pkl", "wb") as f:
    pickle.dump(corpus_tokens, f)

print(f"\n✅ bm25_index.pkl + corpus_tokens.pkl salvos")


🔎 Construindo BM25 Index (rank_bm25)...
  BM25 corpus size : 126
  BM25 avgdl       : 360.84 tokens
  IDF vocab size   : 7,752

✅ bm25_index.pkl + corpus_tokens.pkl salvos


## 🔍 Seção 7 — Funções de Busca Reutilizáveis

In [7]:
def _query_tokens(query: str) -> List[str]:
    return tokenize_pt(query)

def _minmax_norm(scores: np.ndarray) -> np.ndarray:
    mn, mx = scores.min(), scores.max()
    if mx == mn: return np.zeros_like(scores)
    return (scores - mn) / (mx - mn)

def search_tfidf(query: str, top_k: int = 10) -> pd.DataFrame:
    """Busca TF-IDF por similaridade coseno. Retorna top_k documentos."""
    q_vec = tfidf_vectorizer.transform([tokens_to_str(_query_tokens(query))])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = scores.argsort()[::-1][:top_k]
    return pd.DataFrame({
        "rank":       range(1, len(top_idx) + 1),
        "doc_index":  top_idx,
        "article_id": df.iloc[top_idx]["article_id"].values,
        "source":     df.iloc[top_idx]["source"].values,
        "title":      df.iloc[top_idx]["title"].values,
        "score_tfidf": scores[top_idx].round(4),
    })

def search_bm25(query: str, top_k: int = 10) -> pd.DataFrame:
    """Busca BM25Okapi. Retorna top_k documentos."""
    q_tokens = _query_tokens(query)
    if not q_tokens:
        return pd.DataFrame(columns=["rank","doc_index","article_id","source","title","score_bm25"])
    scores = np.array(bm25_index.get_scores(q_tokens))
    top_idx = scores.argsort()[::-1][:top_k]
    return pd.DataFrame({
        "rank":      range(1, len(top_idx) + 1),
        "doc_index": top_idx,
        "article_id": df.iloc[top_idx]["article_id"].values,
        "source":    df.iloc[top_idx]["source"].values,
        "title":     df.iloc[top_idx]["title"].values,
        "score_bm25": scores[top_idx].round(4),
    })

def search_hybrid(query: str, top_k: int = 10, alpha: float = 0.4) -> pd.DataFrame:
    """Hybrid BM25 + TF-IDF. alpha=0.4→TF-IDF, (1-alpha)=0.6→BM25."""
    q_tokens = _query_tokens(query)
    q_str    = tokens_to_str(q_tokens)

    # TF-IDF scores
    q_vec    = tfidf_vectorizer.transform([q_str]) if q_str.strip() else None
    scores_tfidf = cosine_similarity(q_vec, tfidf_matrix).flatten() if q_vec is not None else np.zeros(len(df))

    # BM25 scores
    scores_bm25 = np.array(bm25_index.get_scores(q_tokens)) if q_tokens else np.zeros(len(df))

    # MinMax normalize & combine
    s_tf  = _minmax_norm(scores_tfidf)
    s_bm  = _minmax_norm(scores_bm25)
    combo = alpha * s_tf + (1 - alpha) * s_bm

    top_idx = combo.argsort()[::-1][:top_k]
    return pd.DataFrame({
        "rank":        range(1, len(top_idx) + 1),
        "doc_index":   top_idx,
        "article_id":  df.iloc[top_idx]["article_id"].values,
        "source":      df.iloc[top_idx]["source"].values,
        "title":       df.iloc[top_idx]["title"].values,
        "score_tfidf": scores_tfidf[top_idx].round(4),
        "score_bm25":  scores_bm25[top_idx].round(4),
        "score_hybrid": combo[top_idx].round(4),
    })

_SEARCH_LINES = [
    "import pickle, numpy as np, pandas as pd",
    "from sklearn.metrics.pairwise import cosine_similarity",
    "from pathlib import Path",
    "",
    "_DIR  = Path(__file__).parent",
    "_vect = pickle.load(open(_DIR / 'tfidf_vectorizer.pkl','rb'))",
    "_bm25 = pickle.load(open(_DIR / 'bm25_index.pkl','rb'))",
    "_map  = pd.read_parquet(_DIR / 'doc_index_map.parquet')",
    "",
    "def _tok(q):",
    "    import re, unicodedata",
    "    t2 = unicodedata.normalize('NFD',q).encode('ascii','ignore').decode().lower()",
    "    return [w for w in re.findall(r'[a-z]{3,}',t2) if len(w)>=3]",
    "",
    "def search_bm25(query, top_k=10):",
    "    sc = np.array(_bm25.get_scores(_tok(query)))",
    "    idx = sc.argsort()[::-1][:top_k]",
    "    r = _map.iloc[idx].copy(); r['score_bm25']=sc[idx]; r['rank']=range(1,len(idx)+1)",
    "    return r",
    "",
    "def search_tfidf(query, top_k=10):",
    "    from scipy.sparse import load_npz",
    "    qv = _vect.transform([' '.join(_tok(query))])",
    "    mat = load_npz(_DIR/'tfidf_matrix.npz')",
    "    sc = cosine_similarity(qv,mat).flatten()",
    "    idx = sc.argsort()[::-1][:top_k]",
    "    r = _map.iloc[idx].copy(); r['score_tfidf']=sc[idx]; r['rank']=range(1,len(idx)+1)",
    "    return r",
]
(GOLD_INDEX_DIR / "search_functions.py").write_text("\n".join(_SEARCH_LINES))

print("✅ Funções de busca definidas: search_tfidf | search_bm25 | search_hybrid")
print("✅ search_functions.py salvo em data/gold/tfidf_bm25/")

✅ Funções de busca definidas: search_tfidf | search_bm25 | search_hybrid
✅ search_functions.py salvo em data/gold/tfidf_bm25/


## ✅ Seção 8 — Smoke Tests de Busca

In [8]:

QUERIES = [
    "dividend yield fundo imobiliário",
    "vacância risco inadimplência",
    "FII tijolo logística galpão",
    "CSHG11 HGLG11 KNRI11",
    "análise fundamentalista FII papel",
]

print("\n🔍 Smoke Tests de Busca\n" + "=" * 70)
for q in QUERIES:
    res_bm25   = search_bm25(q, top_k=3)
    res_tfidf  = search_tfidf(q, top_k=3)
    res_hybrid = search_hybrid(q, top_k=3)
    print(f"\n📌 Query: '{q}'")
    print(f"  BM25   top1: {res_bm25.iloc[0]['title'][:75] if len(res_bm25) else 'N/A'}")
    print(f"  TF-IDF top1: {res_tfidf.iloc[0]['title'][:75] if len(res_tfidf) else 'N/A'}")
    print(f"  Hybrid top1: {res_hybrid.iloc[0]['title'][:75] if len(res_hybrid) else 'N/A'}")


🔍 Smoke Tests de Busca

📌 Query: 'dividend yield fundo imobiliário'
  BM25   top1: Fundos imobiliários de papel lideram distribuição de dividendos em 2026; co
  TF-IDF top1: Fundos imobiliários de papel lideram distribuição de dividendos em 2026; co
  Hybrid top1: Fundos imobiliários de papel lideram distribuição de dividendos em 2026; co

📌 Query: 'vacância risco inadimplência'
  BM25   top1: Fiagro: como investir no agronegócio
  TF-IDF top1: MXRF11, HGLG11, BTLG11 e mais fundos imobiliários com atualizações esta sem
  Hybrid top1: MXRF11, HGLG11, BTLG11 e mais fundos imobiliários com atualizações esta sem

📌 Query: 'FII tijolo logística galpão'
  BM25   top1: Mercado Ações FIIs Ferramentas Notícias IRPF Imposto de Renda 2026 Guia do 
  TF-IDF top1: FIIs em destaque Papel: Papéis MXRF11 9,72 -0,31% 12,26% Papel Papel Fiagro
  Hybrid top1: FIIs em destaque Papel: Papéis MXRF11 9,72 -0,31% 12,26% Papel Papel Fiagro

📌 Query: 'CSHG11 HGLG11 KNRI11'
  BM25   top1: MXRF11, HGLG11, BTLG11

## 📋 Seção 9 — Tabela de Relevância por Doc (Gold)

In [9]:

print("\n📋 Gerando tabela de relevância por documento...")

FII_QUERIES = [
    "fundo imobiliário dividendo yield",
    "vacância risco galpão logística",
    "shopping laje corporativa renda",
    "FII papel CRI CRA debentures",
    "inadimplência calote fundo",
]

rows = []
for qi, query in enumerate(FII_QUERIES):
    res = search_hybrid(query, top_k=20)
    for _, row in res.iterrows():
        rows.append({
            "query":        query,
            "query_index":  qi,
            "rank":         row["rank"],
            "article_id":   row["article_id"],
            "source":       row["source"],
            "title":        row["title"],
            "score_tfidf":  row.get("score_tfidf", 0),
            "score_bm25":   row.get("score_bm25", 0),
            "score_hybrid": row.get("score_hybrid", 0),
        })

df_relevance = pd.DataFrame(rows)
_path = GOLD_INDEX_DIR / "document_relevance.parquet"
df_relevance.to_parquet(_path, index=False, compression="snappy")
print(f"✅ document_relevance.parquet: {len(df_relevance):,} linhas")


📋 Gerando tabela de relevância por documento...
✅ document_relevance.parquet: 100 linhas


## 📊 Seção 10 — Resumo de Outputs

In [10]:

print("\n📊 OUTPUTS NB04 — data/gold/tfidf_bm25/")
print("=" * 65)
artefatos = [
    ("tfidf_vectorizer.pkl",      "Vectorizer treinado (sklearn)"),
    ("tfidf_matrix.npz",          "Matriz TF-IDF sparse (N_docs × V)"),
    ("bm25_index.pkl",            "Índice BM25Okapi (rank_bm25)"),
    ("corpus_tokens.pkl",         "Corpus tokenizado (List[List[str]])"),
    ("doc_index_map.parquet",     "Mapeamento doc_index ↔ article_id"),
    ("document_relevance.parquet","Relevância por query × doc"),
    ("search_functions.py",       "Módulo de busca reutilizável"),
]
for fname, desc in artefatos:
    p = GOLD_INDEX_DIR / fname
    if p.exists():
        size = p.stat().st_size // 1024
        print(f"  ✅ {fname:<36} {size:>6} KB — {desc}")
    else:
        print(f"  ❌ {fname} — não gerado")

log_quality_check(
    logger,
    nb="NB04",
    tfidf_docs=tfidf_matrix.shape[0],
    vocab=tfidf_matrix.shape[1],
    bm25_avgdl=round(bm25_index.avgdl, 2),
)


📊 OUTPUTS NB04 — data/gold/tfidf_bm25/
  ✅ tfidf_vectorizer.pkl                    380 KB — Vectorizer treinado (sklearn)
  ✅ tfidf_matrix.npz                        159 KB — Matriz TF-IDF sparse (N_docs × V)
  ✅ bm25_index.pkl                          439 KB — Índice BM25Okapi (rank_bm25)
  ✅ corpus_tokens.pkl                       446 KB — Corpus tokenizado (List[List[str]])
  ✅ doc_index_map.parquet                    22 KB — Mapeamento doc_index ↔ article_id
  ✅ document_relevance.parquet               18 KB — Relevância por query × doc
  ✅ search_functions.py                       1 KB — Módulo de busca reutilizável
2026-06-14T15:48:16 | INFO     | fii_pipeline.ingestion | QUALITY_CHECK | nb=NB04 | tfidf_docs=126 | vocab=9449 | bm25_avgdl=360.84



## ✅ NB04 Completo

| Artefato | Consumido por |
|----------|---------------|
| `tfidf_vectorizer.pkl` + `tfidf_matrix.npz` | NB05, NB06 |
| `bm25_index.pkl` + `corpus_tokens.pkl` | NB05, NB06 |
| `doc_index_map.parquet` | NB05, NB06, NB07 |
| `document_relevance.parquet` | NB06, NB07 |
| `search_functions.py` | `api/`, `dashboard/` |

▶️ **Próximo:** `05_contextual_sentiment.ipynb`